# NN Ensembler and Param Testbed

In [4]:
import os
if 'experiments' in os.getcwd ():
    os.chdir (os.getcwd () + "/..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [5]:
####### DATA PATHS #######
TRAIN_PATH = "./data/in/cattle_data_train.csv"
TEST_PATH = "./data/in/cattle_data_test.csv"
OUT_PATH = "./data/out/nnet_ensemble.csv"

# Preprocessing

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

pd.set_option ("display.max_columns", None)
TARGET_FEATURE = "Milk_Yield_L"

DROP_FEATURES = [
    "Cattle_ID",
    "Farm_ID",
    "Feed_Quantity_lb",
    "Breed",
    "Climate_Zone",
    "Management_System",
    "Feed_Type",
    "Feeding_Frequency",
    "Walking_Distance_km",
    "Grazing_Duration_hrs",
    "Rumination_Time_hrs",
    "Resting_Hours",
    "Body_Condition_Score",
    "Humidity_percent",
    "BVD_Vaccine",
    "FMD_Vaccine",
    "Brucellosis_Vaccine",
    "HS_Vaccine",
    "BQ_Vaccine",
    "Housing_Score",
]

CATEGORICAL_FEATURES = [
    "Lactation_Stage",
    "Date",
    "Milking_Interval_hrs",
]

STANDARD_SCALED_FEATURES = [
    "Age_Months",
    "Weight_kg",
    "Parity",
    "Days_in_Milk",
    "Feed_Quantity_kg",
    "Water_Intake_L",
    "Ambient_Temperature_C",
    "Previous_Week_Avg_Yield",
]

def preprocess (dtrain, dtest):
    # Convert month to season
    def month_to_season (m):
        if m in [12, 1, 2]:
            return "Winter"
        elif m in [3, 4, 5]:
            return "Spring"
        elif m in [6, 7, 8]:
            return "Summer"
        else:
            return "Fall"

    months = pd.to_datetime (dtest['Date']).dt.month
    dtest = dtest.drop (columns = ['Date'])
    dtest['Date'] = months.apply (month_to_season)

    months = pd.to_datetime (dtrain['Date']).dt.month
    dtrain = dtrain.drop (columns = ['Date'])
    dtrain['Date'] = months.apply (month_to_season)

    # Imputation
    median_val = dtrain["Feed_Quantity_kg"].median ()

    dtrain.loc[dtrain["Feed_Quantity_kg"].isna (), "Feed_Quantity_kg"] = median_val
    dtest.loc[dtest["Feed_Quantity_kg"].isna (), "Feed_Quantity_kg"] = median_val

    # Drop features deemed unnecessary
    dtrain = dtrain.drop (DROP_FEATURES, axis = 1)
    dtest = dtest.drop (DROP_FEATURES, axis = 1)

    # One-hot encode
    dtrain = pd.get_dummies (dtrain, columns = CATEGORICAL_FEATURES, 
                             drop_first = True)
    dtest = pd.get_dummies (dtest, columns = CATEGORICAL_FEATURES, 
                            drop_first = True)
    dtrain, dtest = dtrain.align (dtest, join = 'left', axis = 1, fill_value = 0)

    # Standardize data
    scaler = StandardScaler ()
    dtrain[STANDARD_SCALED_FEATURES] = scaler.fit_transform (
                                                dtrain[STANDARD_SCALED_FEATURES])
    dtest[STANDARD_SCALED_FEATURES] = scaler.transform (
                                                dtest[STANDARD_SCALED_FEATURES])

    return dtrain, dtest, scaler

train_data = pd.read_csv (TRAIN_PATH)
X_train, X_test, y_train, y_test = train_test_split (
    train_data.drop (TARGET_FEATURE, axis = 1),
    train_data[TARGET_FEATURE], test_size = 0.2, random_state = 0)

X_train, X_test, scaler = preprocess (X_train, X_test)

nan_cols = X_train.columns[X_train.isna ().any ()]
print(f"Features with NaN: {list(nan_cols)}")

Features with NaN: []


# Training Ensemble

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category = ConvergenceWarning)

# Define ensemble configurations - varying architecture, regularization, and random seeds
ensemble_configs = [
    {'hidden_layer_sizes': (128, 96, 64), 'alpha': 0.0001, 'random_state': 0},
    {'hidden_layer_sizes': (85, 85, 85), 'alpha': 0.0001, 'random_state': 0},
    {'hidden_layer_sizes': (115, 115, 115), 'alpha': 0.0001, 'random_state': 0},
    {'hidden_layer_sizes': (160, 120, 80), 'alpha': 0.0001, 'random_state': 0},
    {'hidden_layer_sizes': (100, 100, 100, 100), 'alpha': 0.0001, 'random_state': 0},
    {'hidden_layer_sizes': (64, 64, 64, 64), 'alpha': 0.0001, 'random_state': 0},
    {'hidden_layer_sizes': (64, 64, 64, 64, 64), 'alpha': 0.0001, 'random_state': 0},
    {'hidden_layer_sizes': (200, 200), 'alpha': 0.0001, 'random_state': 0},
]

# Train ensemble
models = []
train_rmse_list = []
test_rmse_list = []
best_test_rmse_list = []  # Track best test RMSE during training
best_iter_list = []  # Track which iteration achieved best test RMSE

total_iterations = 350
iterations_per_step = 10

print(f"Training ensemble of {len(ensemble_configs)} models...")
print()

for idx, config in enumerate(ensemble_configs):
    print(f"Training model {idx+1}/{len(ensemble_configs)}")
    print(f"  Config: hidden_layers={config['hidden_layer_sizes']}, alpha={config['alpha']}, seed={config['random_state']}")
    
    model = MLPRegressor(
        hidden_layer_sizes=config['hidden_layer_sizes'],
        activation="tanh",
        learning_rate_init=0.00003,
        learning_rate="adaptive",
        alpha=config['alpha'],
        early_stopping=False,
        n_iter_no_change=20,
        verbose=False,
        warm_start=True,
        max_iter=iterations_per_step,
        random_state=config['random_state']
    )
    
    # Track best test RMSE during training
    best_test_rmse = float('inf')
    best_iter = 0
    
    # Iterative training with evaluation at each step
    for i in range(iterations_per_step, total_iterations + 1, iterations_per_step):
        model.fit(X_train, y_train)
        
        # Evaluate at this iteration
        y_test_pred = model.predict(X_test)
        current_test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
        
        # Track best
        if current_test_rmse < best_test_rmse:
            best_test_rmse = current_test_rmse
            best_iter = i
    
    # Final evaluation
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    
    print(f"  Final - Train RMSE: {train_rmse:.4f}, Test RMSE: {test_rmse:.4f}")
    print(f"  Best test RMSE: {best_test_rmse:.4f} (at iteration {best_iter})")
    if test_rmse > best_test_rmse:
        print(f"Overfitting detected: Test RMSE increased by {test_rmse - best_test_rmse:.4f}")
    print()
    
    models.append(model)
    train_rmse_list.append(train_rmse)
    test_rmse_list.append(test_rmse)
    best_test_rmse_list.append(best_test_rmse)
    best_iter_list.append(best_iter)

# Ensemble predictions (simple averaging)
print("=" * 60)
print("ENSEMBLE RESULTS")
print("=" * 60)

train_preds = np.array([model.predict(X_train) for model in models])
test_preds = np.array([model.predict(X_test) for model in models])

ensemble_train_pred = train_preds.mean(axis=0)
ensemble_test_pred = test_preds.mean(axis=0)

ensemble_train_rmse = np.sqrt(mean_squared_error(y_train, ensemble_train_pred))
ensemble_test_rmse = np.sqrt(mean_squared_error(y_test, ensemble_test_pred))

print(f"\nIndividual model performance:")
print(f"  Final test RMSE - Best: {min(test_rmse_list):.4f}, Worst: {max(test_rmse_list):.4f}, Mean: {np.mean(test_rmse_list):.4f}")
print(f"  Best achieved test RMSE - Best: {min(best_test_rmse_list):.4f}, Worst: {max(best_test_rmse_list):.4f}, Mean: {np.mean(best_test_rmse_list):.4f}")
print(f"  Average overfitting (final - best): {np.mean([test_rmse_list[i] - best_test_rmse_list[i] for i in range(len(models))]):.4f}")

print(f"\nEnsemble performance:")
print(f"  Ensemble Train RMSE: {ensemble_train_rmse:.4f}")
print(f"  Ensemble Test RMSE:  {ensemble_test_rmse:.4f}")
print(f"  Improvement over best final individual: {min(test_rmse_list) - ensemble_test_rmse:.4f}")
print(f"  Improvement over best achieved individual: {min(best_test_rmse_list) - ensemble_test_rmse:.4f}")

Training ensemble of 8 models...

Training model 1/8
  Config: hidden_layers=(128, 96, 64), alpha=0.0001, seed=0
  Final - Train RMSE: 4.0871, Test RMSE: 4.1131
  Best test RMSE: 4.1098 (at iteration 190)
  ⚠️  Overfitting detected: Test RMSE increased by 0.0033

Training model 2/8
  Config: hidden_layers=(85, 85, 85), alpha=0.0001, seed=0
  Final - Train RMSE: 4.0943, Test RMSE: 4.1134
  Best test RMSE: 4.1121 (at iteration 250)
  ⚠️  Overfitting detected: Test RMSE increased by 0.0013

Training model 3/8
  Config: hidden_layers=(115, 115, 115), alpha=0.0001, seed=0
  Final - Train RMSE: 4.0904, Test RMSE: 4.1145
  Best test RMSE: 4.1127 (at iteration 230)
  ⚠️  Overfitting detected: Test RMSE increased by 0.0018

Training model 4/8
  Config: hidden_layers=(160, 120, 80), alpha=0.0001, seed=0
  Final - Train RMSE: 4.0831, Test RMSE: 4.1145
  Best test RMSE: 4.1094 (at iteration 170)
  ⚠️  Overfitting detected: Test RMSE increased by 0.0052

Training model 5/8
  Config: hidden_layers=(

# Analyze Ensemble Predictions

In [8]:
# Analyze best and worst predictions from ensemble
raw_data = pd.read_csv(TRAIN_PATH)

errors = np.sqrt((y_test - ensemble_test_pred) ** 2)
df_results = X_test.copy()
scaled_part = df_results[STANDARD_SCALED_FEATURES]
scaled_inverse = pd.DataFrame(scaler.inverse_transform(scaled_part),
                               columns=STANDARD_SCALED_FEATURES,
                               index=df_results.index)

# Replace only those columns
df_results[STANDARD_SCALED_FEATURES] = scaled_inverse
df_results["y_true"] = y_test
df_results["y_pred"] = ensemble_test_pred
df_results["rmse"] = errors

df_results = df_results.merge(
    raw_data,
    left_index=True,
    right_index=True,
    how="left"
)

print("\nTop 5 BEST predictions:")
print(df_results.nsmallest(5, "rmse"))

print("\nTop 5 WORST predictions:")
print(df_results.nlargest(5, "rmse"))


Top 5 BEST predictions:
        Age_Months_x  Weight_kg_x  Parity_x  Days_in_Milk_x  \
162167          29.0        259.7       4.0            35.0   
52201           95.0        432.9       2.0           361.0   
155958         126.0        652.0       5.0            45.0   
148842          48.0        454.0       4.0           132.0   
24316           63.0        667.8       3.0            16.0   

        Feed_Quantity_kg_x  Water_Intake_L_x  Ambient_Temperature_C_x  \
162167           11.793932         85.985360                23.806155   
52201             5.898166         95.988903                14.267118   
155958           17.859104         90.116359                19.567617   
148842            5.619499        107.891892                24.059736   
24316            14.207062         98.925440                21.882164   

        Anthrax_Vaccine_x  IBR_Vaccine_x  Rabies_Vaccine_x  \
162167                  0              1                 0   
52201                   1        

# Final Model - Train on Full Dataset

In [9]:
# Build final ensemble on full training data
train_data = pd.read_csv(TRAIN_PATH)
test_data = pd.read_csv(TEST_PATH)

X_train_full = train_data.drop(TARGET_FEATURE, axis=1)
y_train_full = train_data[TARGET_FEATURE]
X_test_full = test_data

X_train_full, X_test_full, scaler_full = preprocess(X_train_full, X_test_full)

print(f"Training final ensemble on full dataset...")
print()

final_models = []

for idx, config in enumerate(ensemble_configs):
    print(f"Training final model {idx+1}/{len(ensemble_configs)}")
    
    model = MLPRegressor(
        hidden_layer_sizes=config['hidden_layer_sizes'],
        activation="tanh",
        learning_rate_init=0.00003,
        learning_rate="adaptive",
        alpha=config['alpha'],
        early_stopping=False,
        n_iter_no_change=20,
        verbose=False,
        max_iter=total_iterations,
        random_state=config['random_state']
    )
    
    model.fit(X_train_full, y_train_full)
    final_models.append(model)
    print(f"  Model {idx+1} trained successfully")

print("\nAll models trained!")

Training final ensemble on full dataset...

Training final model 1/8


  Model 1 trained successfully
Training final model 2/8
  Model 2 trained successfully
Training final model 3/8
  Model 3 trained successfully
Training final model 4/8
  Model 4 trained successfully
Training final model 5/8
  Model 5 trained successfully
Training final model 6/8
  Model 6 trained successfully
Training final model 7/8
  Model 7 trained successfully
Training final model 8/8
  Model 8 trained successfully

All models trained!


In [10]:
# Final Predictions - ensemble average
final_preds = np.array([model.predict(X_test_full) for model in final_models])
y_pred_final = final_preds.mean(axis=0)

print(f"Ensemble prediction mean: {y_pred_final.mean():.4f}")
print(f"Ensemble prediction std: {y_pred_final.std():.4f}")
print(f"Prediction range: [{y_pred_final.min():.4f}, {y_pred_final.max():.4f}]")

out_data = pd.DataFrame({
    'Cattle_ID': np.arange(1, len(y_pred_final) + 1),
    'Milk_Yield_L': y_pred_final
})
out_data.to_csv(OUT_PATH, index=False)
print(f"\nPredictions saved to {OUT_PATH}")

Ensemble prediction mean: 15.5881
Ensemble prediction std: 3.4514
Prediction range: [4.9853, 27.6118]

Predictions saved to ./data/out/nnet_ensemble.csv
